# Lesson 9 — Intent routing: classify first, then pick the handler

Lesson 8 covered composite scoring. This lesson covers a different use of `Choice`: routing an incoming request to one of several handlers — deterministic code, a specialist model, or a human — based on what it's actually asking for.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source:</strong> docs.typesafe.ai/patterns/intent-routing (click to expand)</summary>

# Intent routing

> Classify incoming requests and route each to the optimal handler: deterministic logic, a specialist LLM, or a human.

Not every user request needs the same kind of handler. Some can be answered with a database lookup. Some need an LLM with domain-specific context. Some need a human. TypeSafe can sit in front of all of these as a fast, cheap classifier that determines which handler to invoke.

## Example: customer service routing

Let's imagine you are building a customer service system. Messages come in and need to be routed to the right handler. Rather than sending every message through an expensive LLM to figure out what kind of request it is, you classify first and route accordingly.

### Step 1: classify intent and complexity

```json
{
  "intent": {
    "type": "choice",
    "instructions": "The primary intent of this customer message",
    "criteria": {
      "order_status": "Asking about an existing order",
      "product_question": "Asking about a product before buying",
      "return_exchange": "Wants to return or exchange something",
      "complaint": "Unhappy with experience, wants resolution"
    }
  },
  "complexity": {
    "type": "score",
    "instructions": "How complex is this request to resolve",
    "criteria": [
      "Simple lookup or standard procedure",
      "Requires some judgment or multi-step process",
      "Unusual situation, edge case, or escalation needed"
    ]
  }
}
```

### Step 2: route to the optimal handler

```python
# routing.py
def route_ticket(ticket_id, response):
    intent = response.answers["intent"]
    complexity = response.answers["complexity"]

    if intent.confidence < 0.5:
        # If we don't have enough confidence to classify, route to a human agent
        return route_to_human_agent(ticket_id)

    if intent.choice == "order_status":
        handle_order_status(ticket_id)

    elif intent.choice == "product_question":
        handle_with_llm(ticket_id, PRODUCT_SPECIALIST)

    elif intent.choice == "return_exchange":
        handle_with_llm(ticket_id, RETURNS_SPECIALIST)

    elif intent.choice == "complaint":
        low_confidence = complexity.confidence < 0.5
        # A higher complexity.score leans toward the "escalation needed" end of the scale.
        if complexity.score > 1 or low_confidence:
            # Too complex for safe automation, or we're not sure about the complexity; route to a human.
            route_to_human_agent(ticket_id)
        else:
            handle_with_llm(ticket_id, COMPLAINT_RESOLUTION)
```

One intent routes to deterministic code with no LLM involved. Two route to different specialist LLMs, each loaded with different context. One uses the complexity score to decide between an LLM and a human. TypeSafe handles the classification all in a single quick call; the expensive resources only get invoked for the requests that actually need them.

Note the additional confidence check on the complexity score. As discussed in [Confidence](https://docs.typesafe.ai/confidence), it is always important to consider the meaning of a low confidence score in the context of the system and the stakes of the decision.

</details>

## The core idea

Intent routing is `Choice` (lesson 2) aimed specifically at "which handler," composed with confidence-gated routing (lesson 7): classify the intent, check whether the classification itself is trustworthy, and only then dispatch. The handlers themselves aren't all the same kind of thing — deterministic code, a specialist model, or a human — which is the point. TypeSafe's job is just the cheap first triage that decides which expensive resource, if any, gets involved.

The example below adds a second layer: even once you're confident about *what* the customer wants, a `Score` on how complex the request is can still route a low-confidence or high-complexity `complaint` to a human instead of a model.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice, score } from "npm:@typesafe-ai/sdk";

const message =
  "This is the third damaged item in a row and I want to speak to someone about a refund, this is ridiculous.";

const response = await client.systemOne({
  state: message,
  questions: {
    intent: choice("The primary intent of this customer message", {
      orderStatus: "Asking about an existing order",
      productQuestion: "Asking about a product before buying",
      returnExchange: "Wants to return or exchange something",
      complaint: "Unhappy with experience, wants resolution",
    }),
    complexity: score("How complex is this request to resolve", [
      "Simple lookup or standard procedure",
      "Requires some judgment or multi-step process",
      "Unusual situation, edge case, or escalation needed",
    ]),
  },
});

const { intent, complexity } = response.answers;

if (intent.confidence < 0.5) {
  console.log("→ route to a human agent (intent unclear)");
} else if (intent.choice === "orderStatus") {
  console.log("→ handle with deterministic order-status lookup");
} else if (intent.choice === "productQuestion") {
  console.log("→ hand off to the product-specialist model");
} else if (intent.choice === "returnExchange") {
  console.log("→ hand off to the returns-specialist model");
} else {
  const lowConfidence = complexity.confidence < 0.5;
  if (complexity.score > 1 || lowConfidence) {
    console.log("→ too complex (or too uncertain) for automation, route to a human");
  } else {
    console.log("→ hand off to the complaint-resolution model");
  }
}

## What just happened

This message is a `complaint` almost by definition ("third damaged item," "ridiculous"), so the interesting question is `complexity`, not `intent` — a repeated failure with an explicit refund request reads as more than a simple lookup, which is exactly the case the docs' own routing function treats as too much for automation.

**Try it yourself:** predict `intent` and `complexity` for a plain "where's my order #4471" message before running it — expect a confident `orderStatus` and low `complexity`. Then predict what happens to a message that's genuinely ambiguous between `product_question` and `complaint` (e.g. "is this supposed to break this easily?"), and check whether `intent.confidence` drops below the `0.5` floor.

## Next up

That's all four architectural patterns from `docs.typesafe.ai/patterns` covered — speculative fan-out, confidence-gated routing, composite scoring, intent routing. From here the material moves into the `cookbooks/` tree: worked, difficulty-tagged examples across self-consistency, batching, how-to, extraction, and classification.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.